# NumPy Interview Refresher

Build fast recall for shapes, combining, broadcasting, indexing, vectorization, ranking, linear algebra, and ML primitives.

- **Study time:** 45-55 minutes
- **Prerequisites:** basic Python expressions and loops
- **Mode:** `quick`
- **Data policy:** no external files or downloads; seeded synthetic arrays only
- **Provenance:** consolidated from the legacy NumPy refresher variants and advanced curated cells

Output convention: every retained textual result begins with a label that identifies the operation that produced it.
Annotation convention: comments explain intent, shape changes, invariants, subtle API behavior, or configuration side effects; obvious Python syntax is left uncommented.


In [1]:
import numpy as np

np.set_printoptions(
    precision=3, suppress=True
)  # Display only: 3-digit precision and no scientific notation; underlying values are unchanged.
rng = np.random.default_rng(42)  # Reproducible local generator without global RNG side effects.

print("Environment | NumPy version", np.__version__)

Environment | NumPy version 2.5.2


## 1. Creation and dtype

Predict each shape and dtype before running the cell. An ML implementation can silently fail when an integer array truncates a floating-point update.


In [2]:
vector = np.array([1, 2, 3])
matrix = np.array([[1, 2], [3, 4]], dtype=np.float32)
float_vector = vector.astype(
    np.float64
)  # Promote before operations whose fractional results must survive.
zeros = np.zeros((2, 3))
identity = np.eye(3)
samples = rng.normal(size=(3, 4))

print("Creation | vector (value, dtype, shape)", (vector, vector.dtype, vector.shape))
print(
    "Creation | matrix metadata",
    {"shape": matrix.shape, "ndim": matrix.ndim, "size": matrix.size, "dtype": matrix.dtype},
)
print("Casting | integer vector to floating dtype", float_vector.dtype)
print("Creation | zeros (2 x 3)", zeros)
print("Creation | identity matrix", identity)
print("Creation | seeded normal samples", samples)

Creation | vector (value, dtype, shape) (array([1, 2, 3]), dtype('int64'), (3,))
Creation | matrix metadata {'shape': (2, 2), 'ndim': 2, 'size': 4, 'dtype': dtype('float32')}
Casting | integer vector to floating dtype float64
Creation | zeros (2 x 3) [[0. 0. 0.]
 [0. 0. 0.]]
Creation | identity matrix [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
Creation | seeded normal samples [[ 0.305 -1.04   0.75   0.941]
 [-1.951 -1.302  0.128 -0.316]
 [-0.017 -0.853  0.879  0.778]]


## 2. Shape changes, views, and copies


In [3]:
base = np.arange(12)
reshaped = base.reshape(3, 4)
raveled = reshaped.ravel()  # View when memory layout permits.
flattened = reshaped.flatten()  # Independent copy.
transposed = reshaped.T  # Axis swap; usually a view.
expanded_front = reshaped[None, :, :]  # Insert batch axis: (1, 3, 4).
expanded_last = np.expand_dims(reshaped, -1)  # Insert trailing axis: (3, 4, 1).
squeezed = np.squeeze(expanded_last, axis=-1)  # Remove only the named size-1 axis.

print("Shape | base -> reshaped", f"{base.shape} -> {reshaped.shape}")
print("Memory | ravel shares memory", np.shares_memory(reshaped, raveled))
print("Memory | flatten shares memory", np.shares_memory(reshaped, flattened))
print("Shape/memory | transpose", (transposed.shape, np.shares_memory(reshaped, transposed)))
print(
    "Shape | add/remove singleton axes",
    (expanded_front.shape, expanded_last.shape, squeezed.shape),
)

Shape | base -> reshaped (12,) -> (3, 4)
Memory | ravel shares memory True
Memory | flatten shares memory False
Shape/memory | transpose ((4, 3), True)
Shape | add/remove singleton axes ((1, 3, 4), (3, 4, 1), (3, 4))


## 3. Combining and splitting arrays

`concatenate` joins along an existing axis, so every other dimension must match. `stack` inserts a new axis, so every input shape must match. `vstack` and `hstack` are conveniences; prefer an explicit axis when 1-D behavior could be ambiguous.


In [4]:
left = np.arange(9).reshape(3, 3)
right = left + 10

concatenated_rows = np.concatenate([left, right], axis=0)  # Extend rows: (6, 3).
concatenated_columns = np.concatenate([left, right], axis=1)  # Extend columns: (3, 6).
stacked_axis0 = np.stack([left, right], axis=0)  # Insert before rows: (2, 3, 3).
stacked_axis1 = np.stack([left, right], axis=1)  # Insert between rows/columns: (3, 2, 3).
stacked_axis2 = np.stack([left, right], axis=2)  # Insert after columns: (3, 3, 2).
equal_halves = np.split(concatenated_rows, 2, axis=0)  # Requires an exact division.
uneven_chunks = np.array_split(np.arange(7), 4)  # Allows chunk sizes to differ by one.

print("Combine | input shapes", (left.shape, right.shape))
print(
    "Combine | concatenate along existing axes",
    {"axis=0": concatenated_rows.shape, "axis=1": concatenated_columns.shape},
)
print(
    "Combine | stack along new axes",
    {
        "axis=0": stacked_axis0.shape,
        "axis=1": stacked_axis1.shape,
        "axis=2": stacked_axis2.shape,
    },
)
print(
    "Convenience | vstack/hstack equal explicit concatenate",
    (
        np.array_equal(np.vstack([left, right]), concatenated_rows),
        np.array_equal(np.hstack([left, right]), concatenated_columns),
    ),
)
print(
    "Split | equal and uneven chunk shapes",
    ([part.shape for part in equal_halves], [part.shape for part in uneven_chunks]),
)

Combine | input shapes ((3, 3), (3, 3))
Combine | concatenate along existing axes {'axis=0': (6, 3), 'axis=1': (3, 6)}
Combine | stack along new axes {'axis=0': (2, 3, 3), 'axis=1': (3, 2, 3), 'axis=2': (3, 3, 2)}
Convenience | vstack/hstack equal explicit concatenate (True, True)
Split | equal and uneven chunk shapes ([(3, 3), (3, 3)], [(2,), (2,), (2,), (1,)])


## 4. Slicing, boolean masks, and fancy indexing


In [5]:
A = np.arange(20).reshape(4, 5)
sliced = A[:2, 1:4]  # Basic slicing preserves a view into A.
mask = A % 3 == 0
selected = A[mask]  # Boolean indexing returns a compact 1-D copy.
rows = np.array([0, 2, 3])
columns = np.array([1, 4, 0])
paired = A[rows, columns]  # Pair coordinates elementwise, not as a Cartesian product.
binary = np.where(A > 12, 1, 0)
where_rows, where_columns = np.where(A > 12)
where_coordinates = np.column_stack([where_rows, where_columns])  # Shape: (n_matches, 2).

print("Indexing | source A", A)
print("Indexing | basic slice A[:2, 1:4]", sliced)
print("Memory | slice shares memory with A", np.shares_memory(A, sliced))
print("Indexing | boolean mask dtype and shape", (mask.dtype, mask.shape))
print("Indexing | values divisible by three", selected)
print("Indexing | paired fancy selection A[rows, columns]", paired)
print("Selection | np.where(A > 12, 1, 0)", binary)
print("Selection | coordinates returned by np.where(condition)", where_coordinates)

Indexing | source A [[ 0  1  2  3  4]
 [ 5  6  7  8  9]
 [10 11 12 13 14]
 [15 16 17 18 19]]
Indexing | basic slice A[:2, 1:4] [[1 2 3]
 [6 7 8]]
Memory | slice shares memory with A True
Indexing | boolean mask dtype and shape (dtype('bool'), (4, 5))
Indexing | values divisible by three [ 0  3  6  9 12 15 18]
Indexing | paired fancy selection A[rows, columns] [ 1 14 15]
Selection | np.where(A > 12, 1, 0) [[0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 1 1]
 [1 1 1 1 1]]
Selection | coordinates returned by np.where(condition) [[2 3]
 [2 4]
 [3 0]
 [3 1]
 [3 2]
 [3 3]
 [3 4]]


## 5. Broadcasting

Align shapes from the right. Adding singleton axes turns pairwise operations into ordinary elementwise arithmetic.


In [6]:
X = rng.normal(size=(5, 2))  # (n=5, d=2)
centers = rng.normal(size=(3, 2))  # (k=3, d=2)
mean = X.mean(axis=0, keepdims=True)  # Keep (1, d) for explicit row-wise broadcasting.
centered = X - mean
differences = X[:, None, :] - centers[None, :, :]  # All sample-center pairs: (n, k, d).
squared_distances = np.sum(differences**2, axis=2)  # Reduce features: (n, k).

print("Broadcast | X, mean, centered shapes", (X.shape, mean.shape, centered.shape))
print("Broadcast | pairwise difference shape", differences.shape)
print("Broadcast | pairwise squared distances (n x k)", squared_distances)

Broadcast | X, mean, centered shapes ((5, 2), (1, 2), (5, 2))
Broadcast | pairwise difference shape (5, 3, 2)
Broadcast | pairwise squared distances (n x k) [[2.98  2.433 0.798]
 [1.067 1.06  1.504]
 [1.376 1.003 1.781]
 [0.129 1.799 0.292]
 [2.258 0.167 1.609]]


## 6. Reductions and standardization


In [7]:
X = rng.normal(loc=100, scale=10, size=(6, 3))
feature_mean = X.mean(axis=0, keepdims=True)  # One statistic per feature.
feature_std = X.std(axis=0, keepdims=True)
standardized = (X - feature_mean) / (feature_std + 1e-12)  # Keep constant-feature divisions finite.

print("Reduction | X.sum(axis=0) shape", X.sum(axis=0).shape)
print("Reduction | X.sum(axis=1) shape", X.sum(axis=1).shape)
print("Standardization | feature means after transform", standardized.mean(axis=0))
print("Standardization | feature std after transform", standardized.std(axis=0))

Reduction | X.sum(axis=0) shape (3,)
Reduction | X.sum(axis=1) shape (6,)
Standardization | feature means after transform [ 0. -0. -0.]
Standardization | feature std after transform [1. 1. 1.]


## 7. Top-k with partial sorting and aligned gathers

`kth` is a zero-based partition position, not a count. For the `k` largest values, partition `-scores` at `k - 1`: negation turns large scores into small partition keys. The first `k` candidates are selected, but their order is not guaranteed—even if a small example happens to look sorted—so sort that candidate set and apply the same order to indices and scores.


In [8]:
scores = np.arange(24).reshape(3, 8)  # Hand-checkable rows keep the ranking mechanics visible.
k = 3
candidate_indices = np.argpartition(-scores, kth=k - 1, axis=1)[
    :, :k
]  # kth is zero-based; candidate order is not guaranteed.
candidate_scores = np.take_along_axis(
    scores, candidate_indices, axis=1
)  # Gather aligned values: (n_rows, k).
candidate_order = np.argsort(
    -candidate_scores, axis=1
)  # Sort only the k candidates, not every full row.
topk_indices = np.take_along_axis(
    candidate_indices, candidate_order, axis=1
)  # Restore original column positions.
topk_scores = np.take_along_axis(
    candidate_scores, candidate_order, axis=1
)  # Apply the same local order; no second gather from the full matrix.

print("Top-k | source scores", scores)
print("Top-k | unordered candidate indices", candidate_indices)
print("Top-k | scores aligned with candidate indices", candidate_scores)
print("Top-k | sorted indices per row", topk_indices)
print("Top-k | aligned sorted scores", topk_scores)

Top-k | source scores [[ 0  1  2  3  4  5  6  7]
 [ 8  9 10 11 12 13 14 15]
 [16 17 18 19 20 21 22 23]]
Top-k | unordered candidate indices [[7 6 5]
 [7 6 5]
 [7 6 5]]
Top-k | scores aligned with candidate indices [[ 7  6  5]
 [15 14 13]
 [23 22 21]]
Top-k | sorted indices per row [[7 6 5]
 [7 6 5]
 [7 6 5]]
Top-k | aligned sorted scores [[ 7  6  5]
 [15 14 13]
 [23 22 21]]


## 8. Linear algebra: solve, norms, and SVD

With a one-dimensional right-hand side `b`, `solve(A, b)` returns a one-dimensional solution and `A @ solution` is matrix-vector multiplication. Add a singleton axis only when the downstream contract genuinely requires a `(n, 1)` column matrix.


In [9]:
A = rng.normal(size=(4, 4))
b = rng.normal(size=4)
solution = np.linalg.solve(A, b)  # (4, 4) and (4,) -> (4,); avoid forming A^{-1}.
residual = A @ solution - b  # Both terms have shape (4,); no implicit (4, 1) array is created.
U, singular_values, Vt = np.linalg.svd(
    A, full_matrices=False
)  # Compact factors preserve reconstruction.

print("Linear algebra | A, b, solution shapes", (A.shape, b.shape, solution.shape))
print("Linear algebra | solve residual max abs", np.max(np.abs(residual)))
print("Linear algebra | row L2 norms", np.linalg.norm(A, axis=1))
print("Linear algebra | compact SVD shapes", (U.shape, singular_values.shape, Vt.shape))

Linear algebra | A, b, solution shapes ((4, 4), (4,), (4,))
Linear algebra | solve residual max abs 3.3306690738754696e-16
Linear algebra | row L2 norms [1.129 1.645 1.715 2.152]
Linear algebra | compact SVD shapes ((4, 4), (4,), (4, 4))


## 9. Stable softmax

Subtracting each row maximum leaves softmax probabilities unchanged while preventing large logits from overflowing `exp`.


In [10]:
def softmax(logits):
    logits = np.asarray(logits, dtype=float)
    shifted = logits - logits.max(
        axis=1, keepdims=True
    )  # Stabilize exp without changing probabilities.
    exponentials = np.exp(shifted)
    return exponentials / exponentials.sum(axis=1, keepdims=True)


logits = np.array([[1000.0, 1001.0, 999.0], [1.0, 0.0, -1.0]])
probabilities = softmax(logits)

print("Softmax | probabilities", probabilities)
print("Softmax | row sums", probabilities.sum(axis=1))

Softmax | probabilities [[0.245 0.665 0.09 ]
 [0.665 0.245 0.09 ]]
Softmax | row sums [1. 1.]


## 10. Pairwise cosine similarity

Normalize each row to unit length, then an ordinary matrix product computes every left-versus-right cosine similarity.


In [11]:
left = rng.normal(size=(4, 3))
right = rng.normal(size=(5, 3))
left_unit = left / (
    np.linalg.norm(left, axis=1, keepdims=True) + 1e-12
)  # Normalize rows; epsilon handles zero vectors.
right_unit = right / (np.linalg.norm(right, axis=1, keepdims=True) + 1e-12)
cosine = left_unit @ right_unit.T  # All pairwise cosine similarities: (4, 5).

print("Cosine similarity | output shape", cosine.shape)
print("Cosine similarity | first 2 x 3 block", cosine[:2, :3])

Cosine similarity | output shape (4, 5)
Cosine similarity | first 2 x 3 block [[ 0.367  0.081 -0.148]
 [-0.169  0.704  0.059]]


## 11. Repeated-index accumulation with `np.add.at`

Fancy-indexed `+=` updates a temporary buffer, so repeated destinations do not accumulate reliably. `np.add.at` performs unbuffered in-place updates and applies every `(index, value)` pair.


In [12]:
repeated_indices = np.array([0, 1, 1, 3, 3, 3])
values = np.array([10, 1, 1, 5, 2, 2])

buffered = np.zeros(5, dtype=int)
buffered[repeated_indices] += values  # Repeated destinations are written back only once.

accumulated = np.zeros(5, dtype=int)
np.add.at(
    accumulated, repeated_indices, values
)  # Apply all repeated updates: index 1 gets 1+1; index 3 gets 5+2+2.

print("Scatter-add | (index, value) update pairs", np.column_stack([repeated_indices, values]))
print("Scatter-add | buffered fancy-index result", buffered)
print("Scatter-add | unbuffered np.add.at result", accumulated)

Scatter-add | (index, value) update pairs [[ 0 10]
 [ 1  1]
 [ 1  1]
 [ 3  5]
 [ 3  2]
 [ 3  2]]
Scatter-add | buffered fancy-index result [10  1  0  2  0]
Scatter-add | unbuffered np.add.at result [10  2  0  9  0]


## 12. Sliding windows without manual loops

For a sequence of length `n` and window width `w`, the view has shape `(n - w + 1, w)`. Windows overlap in memory, so treat the result as read-only.


In [13]:
from numpy.lib.stride_tricks import sliding_window_view

sequence = np.arange(10)
window_width = 4
windows = sliding_window_view(sequence, window_shape=window_width)
moving_averages = windows.mean(
    axis=1
)  # Reduce within each window, preserving one value per start position.

print("Sliding window | source sequence", sequence)
print("Sliding window | shape", windows.shape)
print("Sliding window | overlapping rows", windows)
print("Sliding window | moving averages", moving_averages)

Sliding window | source sequence [0 1 2 3 4 5 6 7 8 9]
Sliding window | shape (7, 4)
Sliding window | overlapping rows [[0 1 2 3]
 [1 2 3 4]
 [2 3 4 5]
 [3 4 5 6]
 [4 5 6 7]
 [5 6 7 8]
 [6 7 8 9]]
Sliding window | moving averages [1.5 2.5 3.5 4.5 5.5 6.5 7.5]


## 13. NaN-aware reductions

Ordinary reductions propagate missing floating-point values. Use a `nan*` reduction only when excluding missing observations matches the intended policy; an all-NaN slice still has no meaningful mean.


In [14]:
values_with_nan = np.array([1.0, np.nan, 3.0, np.nan, 5.0])
ordinary_mean = np.mean(values_with_nan)
nan_aware_mean = np.nanmean(values_with_nan)  # Exclude NaNs from both sum and count.

print("NaN-aware reduction | input", values_with_nan)
print("NaN-aware reduction | np.mean", ordinary_mean)
print("NaN-aware reduction | np.nanmean", nan_aware_mean)

NaN-aware reduction | input [ 1. nan  3. nan  5.]
NaN-aware reduction | np.mean nan
NaN-aware reduction | np.nanmean 3.0


## 14. Retrieval drills

Re-type these from a blank cell later: concatenate versus stack, top-k alignment, pairwise distances, stable softmax, repeated-index accumulation, and vectorized binary metrics.


In [15]:
y_true = rng.integers(0, 2, size=100)
y_pred = rng.integers(0, 2, size=100)
true_positive = np.sum((y_true == 1) & (y_pred == 1))
false_positive = np.sum((y_true == 0) & (y_pred == 1))
false_negative = np.sum((y_true == 1) & (y_pred == 0))
# Epsilon gives this compact drill a finite zero-denominator policy.
precision = true_positive / (true_positive + false_positive + 1e-12)
recall = true_positive / (true_positive + false_negative + 1e-12)
f1 = 2 * precision * recall / (precision + recall + 1e-12)

assert np.allclose(probabilities.sum(axis=1), 1.0)
assert topk_indices.shape == (3, 3)
assert np.array_equal(topk_scores, [[7, 6, 5], [15, 14, 13], [23, 22, 21]])
assert squared_distances.shape == (5, 3)
assert concatenated_rows.shape == (6, 3)
assert stacked_axis0.shape == (2, 3, 3)
assert stacked_axis1.shape == (3, 2, 3)
assert stacked_axis2.shape == (3, 3, 2)
assert sum(part.size for part in uneven_chunks) == 7
assert np.array_equal(accumulated, [10, 2, 0, 9, 0])
assert not np.array_equal(buffered, accumulated)
assert windows.shape == (7, 4)
assert np.isnan(ordinary_mean) and nan_aware_mean == 3.0
print("Binary metrics | precision, recall, F1", np.round([precision, recall, f1], 3))
print("Drill checks | status", "all assertions passed")

Binary metrics | precision, recall, F1 [0.551 0.614 0.581]
Drill checks | status all assertions passed
